# MFPT vs m0: Network and Reset Protocol Comparison (Julia)

This notebook compares MFPT as a function of m0 across network types and reset protocols,
using only the Julia package functions from src/VoterResetting.jl.

In [ ]:
using Graphs
using Random
using Statistics
using Printf
using Plots

project_root = isdir(joinpath(pwd(), "src")) ? pwd() : dirname(dirname(pwd()))
include(joinpath(project_root, "src", "VoterResetting.jl"))
using .VoterResetting

# Use explicit imports to avoid name ambiguity with other loaded modules.
import .VoterResetting: ComplexParams, hub_reset, random_node_reset, first_passage_time_complex

figures_dir = joinpath(project_root, "figures", "complex")
mkpath(figures_dir)
println("Figures will be saved to: $figures_dir")

Figures will be saved to: c:\Users\gerar\Desktop\Escola\Universitat\Master\TFM\voter-model-stochastic-resetting\figures\complex


In [ ]:
# Parameters
N = 500
r = 10
m0_values = collect(range(-0.9, 0.9, length=11))
nsamples = 1000
seed = 1234

Random.seed!(seed)

graphs = Dict(
    "ER" => erdos_renyi(N, 0.1),
    "RRG" => random_regular_graph(N, 6),
    "SF" => barabasi_albert(N, 2),
)

protocols = [
    "High-degree reset" => :hub_high,
    "Low-degree reset"  => :hub_low,
    "Random-node reset" => :random_node,
]

println("Setup complete: N=$N, r=$r, points=$(length(m0_values)), nsamples=$nsamples")

In [ ]:
function mfpt_curve_vs_m0_complex(G, m0_values; r=0.1, nsamples=1000, protocol::Symbol=:hub_high)
    mfpt = zeros(length(m0_values))
    total = length(m0_values)
    t0 = time()

    for (i, m0) in enumerate(m0_values)
        params = VoterResetting.ComplexParams(r, Float64(m0))

        reset_protocol = if protocol == :hub_high
            VoterResetting.hub_reset(Float64(m0); highest=true)
        elseif protocol == :hub_low
            VoterResetting.hub_reset(Float64(m0); highest=false)
        elseif protocol == :random_node
            VoterResetting.random_node_reset(Float64(m0))
        else
            error("Unknown protocol symbol: $protocol")
        end

        out = VoterResetting.first_passage_time_complex(
            G,
            params;
            consensus_type=:positive,
            nsamples=nsamples,
            reset=reset_protocol,
        )
        mfpt[i] = out.mean_fpt

        elapsed = time() - t0
        rate = i / max(elapsed, 1e-9)
        eta = (total - i) / max(rate, 1e-9)
        @printf("[m0 %d/%d] m0=%.3f | MFPT=%.4f | elapsed=%.1fs | eta=%.1fs\n", i, total, m0, mfpt[i], elapsed, eta)
        flush(stdout)
    end

    return mfpt
end

Setup complete: N=500, r=10, points=11, nsamples=1000


mfpt_curve_vs_m0_complex (generic function with 1 method)

In [ ]:
results = Dict{String, Dict{String, Vector{Float64}}}()

for (gname, G) in graphs
    results[gname] = Dict{String, Vector{Float64}}()
    for (pname, psym) in protocols
        println("Running: $gname | $pname")
        results[gname][pname] = mfpt_curve_vs_m0_complex(
            G,
            m0_values;
            r=r,
            nsamples=nsamples,
            protocol=psym,
        )
    end
end

println("All scans finished.")

Running: RRG | Low-degree reset
[m0 1/11] m0=-0.900 | MFPT=5086.2811 | elapsed=61.0s | eta=609.7s
[m0 2/11] m0=-0.720 | MFPT=1617.4298 | elapsed=99.0s | eta=445.4s
[m0 3/11] m0=-0.540 | MFPT=827.2427 | elapsed=129.2s | eta=344.4s
[m0 4/11] m0=-0.360 | MFPT=547.7247 | elapsed=158.8s | eta=277.9s
[m0 5/11] m0=-0.180 | MFPT=341.0059 | elapsed=177.6s | eta=213.1s
[m0 6/11] m0=0.000 | MFPT=243.3789 | elapsed=192.6s | eta=160.5s
[m0 7/11] m0=0.180 | MFPT=175.0288 | elapsed=203.7s | eta=116.4s
[m0 8/11] m0=0.360 | MFPT=119.3997 | elapsed=211.5s | eta=79.3s
[m0 9/11] m0=0.540 | MFPT=78.0260 | elapsed=216.5s | eta=48.1s
[m0 10/11] m0=0.720 | MFPT=42.0266 | elapsed=218.7s | eta=21.9s
[m0 11/11] m0=0.900 | MFPT=17.2763 | elapsed=219.3s | eta=0.0s
Running: RRG | High-degree reset
[m0 1/11] m0=-0.900 | MFPT=4802.3375 | elapsed=56.0s | eta=559.7s
[m0 2/11] m0=-0.720 | MFPT=1594.7853 | elapsed=111.4s | eta=501.2s
[m0 3/11] m0=-0.540 | MFPT=842.5043 | elapsed=144.0s | eta=384.1s
[m0 4/11] m0=-0.360 | 

In [ ]:
# Figure 1: protocol effect within each network
p1 = plot(layout=(1, 3), size=(1500, 420), legend=:topright, yscale=:log10)

for (idx, gname) in enumerate(["ER", "RRG", "SF"])
    for (pname, _) in protocols
        plot!(p1[idx], m0_values, results[gname][pname];
            lw=2, marker=:circle, label=pname)
    end
    xlabel!(p1[idx], "m0")
    ylabel!(p1[idx], "MFPT")
    title!(p1[idx], gname)
end

plot!(p1, plot_title="MFPT vs m0 by network (r=$r, N=$N)")
display(p1)

p1_path = joinpath(figures_dir, "julia_mfpt_vs_m0_by_network.png")
savefig(p1, p1_path)
println("Saved Figure 1 to: $p1_path")

# Figure 2: network effect within each protocol
p2 = plot(layout=(1, 3), size=(1500, 420), legend=:topright, yscale=:log10)

for (idx, (pname, _)) in enumerate(protocols)
    for gname in ["ER", "RRG", "SF"]
        plot!(p2[idx], m0_values, results[gname][pname];
            lw=2, marker=:circle, label=gname)
    end
    xlabel!(p2[idx], "m0")
    ylabel!(p2[idx], "MFPT")
    title!(p2[idx], pname)
end

plot!(p2, plot_title="MFPT vs m0 by protocol (r=$r, N=$N)")
display(p2)

p2_path = joinpath(figures_dir, "julia_mfpt_vs_m0_by_protocol.png")
savefig(p2, p2_path)
println("Saved Figure 2 to: $p2_path")